# Random Forest from scratch — NumPy only, checked against scikit-learn

This notebook builds a random forest out of plain NumPy functions: impurity → split search → one tree (with
per-split feature sampling) → bootstrap → forest → out-of-bag score → MDI importance. Each step restates the idea
from the theory README ([`../README.md`](../README.md)), implements it, and checks it — first on the README's
8-row hand-worked example, at the end against `sklearn.ensemble.RandomForest*` on the two CSV datasets.

**How to use it**

1. Run the imports cell first; then run top to bottom (every section only uses functions defined above it).
2. Each "check" cell prints numbers you can compare with the README by hand. Try to predict them first.
3. Knobs marked `# <- try changing this` are safe to play with.

**Theory → code map**

| README section / equation | function | notebook section |
|---|---|---|
| §2 Gini impurity $1-\sum_k p_k^2$, MSE impurity $\overline{y^2}-\bar y^2$ | `gini_impurity`, `mse_impurity`, `node_totals` | 2 |
| §2 best split = largest weighted impurity decrease (sort + cumulative sums) | `find_best_split` (vs `best_threshold_naive`) | 3 |
| §3 step 3: draw `max_features` candidate features **at every split** | `make_numpy_sampler`, `draw_all_features` | 4 |
| §3 step 2–4: grow one deep tree on weighted rows | `fit_tree`, `predict_tree` | 5 |
| §3 step 1 and 5: bootstrap, then average (or vote) | `bootstrap_counts`, `fit_forest`, `predict_forest` | 6 |
| §4.4 out-of-bag estimate | `oob_score` | 6 |
| §4.6 MDI $=\sum_t \frac{N_t}{N}[\,i_t-\frac{N_L}{N_t}i_L-\frac{N_R}{N_t}i_R\,]$, normalised, averaged | `tree_mdi`, `forest_mdi` | 7 |
| §5 the complete example by hand | all of the above | 2, 3, 5, 7 |
| (verification) the same trees as scikit-learn | `sklearn_counts`, `make_sklearn_sampler`, `compare_trees` | 9, 10 |

| Part | Sections | What it shows |
|---|---|---|
| **A. Building blocks** | 1 – 5 | data, impurity, split search, feature sampling, one tree — each with an implementation hurdle |
| **B. The forest** | 6 – 8 | bootstrap, OOB, MDI, memory of deep trees |
| **C. Checking against scikit-learn** | 9 – 10 | bit-exact agreement in the bagging limit and (by replaying scikit-learn's random numbers) with feature sampling; a statistical comparison with our own random numbers |

All data come from `data/good_fit.csv` and `data/bad_fit.csv`, written by `random_forest_lab.ipynb` — **no internet
needed**.

In [1]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 80, "axes.grid": True, "grid.alpha": 0.3})

FEATURE_THRESHOLD = np.float32(1e-7)    # two values closer than this count as equal (scikit-learn's constant)
EPSILON = np.finfo(np.float64).eps      # a node with impurity <= EPSILON is pure

---
# Part A — Building blocks

## 1 · The data

Both CSVs are loaded exactly as written by the library lab and split with the same `train_test_split` calls, so the
numbers here can be compared with that notebook.

> **Implementation hurdle — scikit-learn's trees work in 32-bit floats.** Before fitting, `X` is converted to
> `float32`; thresholds are midpoints of two `float32` values, computed in 64-bit. If we kept `X` in 64-bit, a test
> value lying between the 32- and 64-bit versions of a threshold could go down a different branch. We therefore
> convert once, here, and use `X32` everywhere.

In [2]:
def load_csv(path):
    data = np.loadtxt(path, delimiter=",", skiprows=1)
    return data[:, :-1], data[:, -1]


X_good, y_good = load_csv(Path("data/good_fit.csv"))
X_bad, y_bad = load_csv(Path("data/bad_fit.csv"))
y_bad = y_bad.astype(int)
Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(X_good, y_good, test_size=1 / 3, random_state=0)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_bad, y_bad, test_size=0.3, random_state=0, stratify=y_bad)
Xg_tr32, Xg_te32 = Xg_tr.astype(np.float32), Xg_te.astype(np.float32)
Xb_tr32, Xb_te32 = Xb_tr.astype(np.float32), Xb_te.astype(np.float32)
print("good fit (regression):    ", Xg_tr.shape, "train", Xg_te.shape, "test")
print("bad fit (classification): ", Xb_tr.shape, "train", Xb_te.shape, "test")
print("largest float64 -> float32 change in X:", np.abs(Xg_tr - Xg_tr32).max())
print("good-fit y is a multiple of 1/64 everywhere:", np.all(yg_tr * 64 == np.round(yg_tr * 64)))

good fit (regression):     (1000, 31) train (500, 31) test
bad fit (classification):  (560, 150) train (240, 150) test
largest float64 -> float32 change in X: 2.0904541031541157e-07
good-fit y is a multiple of 1/64 everywhere: True


## 2 · Impurity of a node (README §2.2)

A node holds some training rows, each with a **weight** $w_i$ (in a forest: how many times the bootstrap drew row
$i$). Everything a node needs is a few weighted totals:

* classification: $W=\sum w_i$ and the weighted class counts $c_k=\sum_{i:\,y_i=k} w_i$;
  Gini impurity $= 1-\sum_k (c_k/W)^2$;
* regression: $W$, $S=\sum w_i y_i$, $Q=\sum w_i y_i^2$; MSE impurity $= Q/W-(S/W)^2$ (mean of $y^2$ minus
  squared mean of $y$ = the variance of $y$ in the node).

Targets are stored as a matrix `T`: one-hot class columns for classification, a single column `y` for regression.

**Check:** the README's 8-row example — root Gini 0.46875, right child 0.375.

In [3]:
def gini_impurity(counts, w):
    """Gini of weighted class counts (last axis = classes) with total weight w."""
    return 1.0 - np.sum(counts * counts, axis=-1) / (w * w)


def mse_impurity(s, q, w):
    """Variance of y from weighted sums: s = sum w*y, q = sum w*y^2, w = sum w."""
    return q / w - (s / w) ** 2.0


def node_totals(T, w, rows, task):
    """Weighted totals of the rows in one node."""
    ww = w[rows]
    if task == "classification":
        return {"w": ww.sum(), "counts": ww @ T[rows]}
    y = T[rows, 0]
    return {"w": ww.sum(), "s": np.sum(ww * y), "q": np.sum(ww * y * y)}


def node_impurity(tot, task):
    if task == "classification":
        return gini_impurity(tot["counts"], tot["w"])
    return mse_impurity(tot["s"], tot["q"], tot["w"])


def node_value(tot, task):
    """What a leaf predicts: class fractions (classification) or the weighted mean (regression)."""
    if task == "classification":
        return tot["counts"] / tot["w"]
    return np.array([tot["s"] / tot["w"]])


X_toy = np.array([[1, 2], [2, 4], [3, 6], [4, 8], [5, 3], [6, 1], [7, 5], [8, 7]], dtype=np.float32)
y_toy = np.array([0, 0, 0, 0, 1, 0, 1, 1])
T_toy, w_toy = np.eye(2)[y_toy], np.ones(8)
root = node_totals(T_toy, w_toy, np.arange(8), "classification")
right = node_totals(T_toy, w_toy, np.arange(4, 8), "classification")
print("root : counts", root["counts"], " Gini", node_impurity(root, "classification"))
print("right: counts", right["counts"], " Gini", node_impurity(right, "classification"))
assert np.isclose(node_impurity(root, "classification"), 0.46875) and np.isclose(node_impurity(right, "classification"), 0.375)

root : counts [5. 3.]  Gini 0.46875
right: counts [1. 3.]  Gini 0.375


## 3 · Finding the best split (README §2.3 and §5)

For one feature, sort the node's rows by that feature. A split "first $p$ rows left, rest right" is allowed only
between two **different** values (and if both sides keep at least `min_samples_leaf` rows). Its quality is the
weighted impurity decrease

$$\Delta = i_\text{node} - \frac{W_L}{W}\, i_L - \frac{W_R}{W}\, i_R .$$

$i_\text{node}$ and $W$ are the same for every candidate, so it is enough to maximise a cheaper **proxy**:
$-(W_L i_L + W_R i_R)$ for Gini, and $S_L^2/W_L + S_R^2/W_R$ for MSE (expand $Q/W-(S/W)^2$ and drop the constant
$Q$). The threshold stored is the midpoint between the two neighbouring values.

> **Implementation hurdle — split search cost.** Recomputing both children's impurity from scratch for each of the
> $n-1$ positions costs $O(n)$ each, $O(n^2)$ per feature. Sorting once and using **cumulative sums** (the left
> child's totals at position $p$ are the running sum of the first $p$ rows; the right child's are total − left)
> costs $O(n\log n)$. `best_threshold_naive` is the slow version, kept only to prove the fast one right and to time
> both. `find_best_split` does all candidate features at once (one column each) and returns the first best split in
> candidate order — the same tie rule scikit-learn uses.

In [4]:
def find_best_split(X32, T, w, rows, candidates, task, tot, min_samples_leaf=1):
    """Best (feature, threshold) over the candidate features for the node holding `rows`, or None."""
    n = len(rows)
    Xn = X32[np.ix_(rows, candidates)]                         # (n, k)
    order = np.argsort(Xn, axis=0, kind="stable")              # sort every candidate column
    Xs = np.take_along_axis(Xn, order, axis=0)
    ws = w[rows][order]                                        # weights in sorted order, (n, k)
    wl = np.cumsum(ws, axis=0)[:-1]                            # left weight for "first p rows left", p = 1..n-1
    wr = tot["w"] - wl
    if task == "classification":
        Tw = T[rows][order] * ws[:, :, None]                   # (n, k, classes)
        cl = np.cumsum(Tw, axis=0)[:-1]
        cr = tot["counts"] - cl
        proxy = -wr * gini_impurity(cr, wr) - wl * gini_impurity(cl, wl)
    else:
        sl = np.cumsum((w[rows] * T[rows, 0])[order], axis=0)[:-1]
        sr = tot["s"] - sl
        proxy = sl * sl / wl + sr * sr / wr
    p = np.arange(1, n)[:, None]
    allowed = (Xs[1:] > Xs[:-1] + FEATURE_THRESHOLD) & (p >= min_samples_leaf) & (n - p >= min_samples_leaf)
    proxy = np.where(allowed, proxy, -np.inf)
    best = int(np.argmax(proxy.T.ravel()))                     # candidate order first, then position
    k, pos = divmod(best, n - 1)
    if proxy[pos, k] == -np.inf:
        return None
    pos += 1
    threshold = np.float64(Xs[pos - 1, k]) / 2.0 + np.float64(Xs[pos, k]) / 2.0
    return {"feature": candidates[k], "threshold": threshold,
            "left": rows[order[:pos, k]], "right": rows[order[pos:, k]]}


def weighted_decrease(tot, tot_left, tot_right, task, w_all):
    """(N_t/N) * [i_t - (N_L/N_t) i_L - (N_R/N_t) i_R] with weights."""
    i, iL, iR = (node_impurity(t, task) for t in (tot, tot_left, tot_right))
    return tot["w"] / w_all * (i - tot_right["w"] / tot["w"] * iR - tot_left["w"] / tot["w"] * iL)


rows = np.arange(8)
for name, rws in [("root", rows), ("right child", np.arange(4, 8))]:
    tot = node_totals(T_toy, w_toy, rws, "classification")
    split = find_best_split(X_toy, T_toy, w_toy, rws, [0, 1], "classification", tot)
    dec = weighted_decrease(tot, node_totals(T_toy, w_toy, split["left"], "classification"),
                            node_totals(T_toy, w_toy, split["right"], "classification"), "classification", 8.0)
    print(f"{name:11s}: best split x{split['feature'] + 1} <= {split['threshold']}, weighted decrease {dec:.5f}")

root       : best split x1 <= 4.5, weighted decrease 0.28125
right child: best split x2 <= 2.0, weighted decrease 0.18750


In [5]:
def best_threshold_naive(x, y, w):
    """O(n^2) reference for one regression feature: try every position, recompute both children from scratch."""
    order = np.argsort(x, kind="stable")
    x, y, w = x[order], y[order], w[order]
    best, best_thr = -np.inf, None
    for p in range(1, len(x)):
        if x[p] <= x[p - 1] + FEATURE_THRESHOLD:
            continue
        cost = sum(np.sum(w[s] * y[s] ** 2) - np.sum(w[s] * y[s]) ** 2 / np.sum(w[s]) for s in (slice(0, p), slice(p, None)))
        if -cost > best:
            best, best_thr = -cost, np.float64(x[p - 1]) / 2 + np.float64(x[p]) / 2
    return best_thr


rng = np.random.default_rng(0)
print("     n   naive O(n^2) [ms]   sort+cumsum O(n log n) [ms]   same threshold?")
for n in [1000, 4000, 16000]:                                  # <- try 32000 (slow!)
    x = rng.normal(size=(n, 1)).astype(np.float32)
    y = np.round(np.sin(3 * x[:, 0]) * 64) / 64 + rng.normal(size=n)
    w = np.ones(n)
    t0 = time.perf_counter(); thr_naive = best_threshold_naive(x[:, 0], y, w); t1 = time.perf_counter()
    split = find_best_split(x, y[:, None], w, np.arange(n), [0], "regression",
                            node_totals(y[:, None], w, np.arange(n), "regression")); t2 = time.perf_counter()
    print(f"{n:6d}   {1e3 * (t1 - t0):12.0f}        {1e3 * (t2 - t1):18.2f}              {thr_naive == split['threshold']}")

     n   naive O(n^2) [ms]   sort+cumsum O(n log n) [ms]   same threshold?
  1000              8                      0.13              True
  4000             45                      0.32              True


 16000            358                      1.27              True


> **How to read this**
> * The 8-row example: the root's best split is `x1 <= 4.5` with weighted decrease **0.28125**, the right child's is
>   `x2 <= 2.0` with **0.1875** — README §5 exactly (features are printed 1-based here, like the README).
> * Timing: going from 1,000 to 16,000 rows (16× more) makes the naive search about 45× slower (≈ 8 → ≈ 360 ms) and
>   the sort + cumulative-sum search only about 10× slower (≈ 0.12 → ≈ 1.2 ms), and both find the same threshold.
>   (Timings vary a little from run to run.) A tree runs this
>   search at every node for every candidate feature, so this trick is what makes forests practical.

## 4 · Per-split feature sampling (README §3, step 3)

At **every** node the forest draws features in random order and evaluates the first `max_features` of them. One
detail is easy to miss:

> **Implementation hurdle — lazy drawing.** If every drawn feature happens to be **constant** inside the node (all
> rows have the same value — common in small nodes and with rounded data), there is nothing to split on. scikit-learn
> then keeps drawing beyond `max_features` until it finds at least one non-constant feature (or runs out); its source
> (`_splitter.pyx`) says "At least one drawn features must be non constant". Constant features also do not count as
> real candidates. We implement the same rule, and measure how often it fires.

In [6]:
def make_numpy_sampler(rng, max_features):
    """Per-node candidate drawing with our own NumPy random numbers."""
    def draw(n_known_constants, is_constant):
        visited, candidates = 0, []
        for f in rng.permutation(len(is_constant)):
            if visited >= max_features and candidates:         # enough drawn AND at least one usable
                break
            visited += 1
            if not is_constant[f]:
                candidates.append(int(f))
        draw.constant_draws += visited - len(candidates)         # bookkeeping for the checks below
        draw.extra_draws += max(0, visited - max_features)
        return candidates, 0
    draw.constant_draws, draw.extra_draws = 0, 0
    return draw


def draw_all_features(n_known_constants, is_constant):
    """No sampling (the bagging limit): every non-constant feature, in index order."""
    return [int(f) for f in np.flatnonzero(~is_constant)], 0


# a node where 8 of 10 features are constant, max_features = 2
is_constant = np.array([True] * 8 + [False] * 2)
sampler = make_numpy_sampler(np.random.default_rng(0), max_features=2)
n_drawn = []
for _ in range(10_000):
    before = sampler.extra_draws
    cands, _ = sampler(0, is_constant)
    n_drawn.append(2 + sampler.extra_draws - before)
print("candidates always non-empty:", len(cands) > 0, "| features drawn per node: mean",
      np.mean(n_drawn).round(2), " max", max(n_drawn), " (max_features = 2)")

candidates always non-empty: True | features drawn per node: mean 3.85  max 9  (max_features = 2)


> **How to read this**
> * Even in a node where 8 of the 10 features are constant and `max_features = 2`, the sampler always returns at least
>   one usable feature; it draws 3.85 features on average and up to 9. Without the rule, the node would give up (and
>   become a leaf) whenever both drawn features are constant — (8/10)·(7/9) ≈ 62 % of the time.

## 5 · One tree (README §3, steps 2–4)

`fit_tree` grows a tree depth-first (a node, then its whole left subtree, then its right subtree — the order
scikit-learn uses, so node numbers line up). A node becomes a leaf when it is pure, has fewer than
`min_samples_split` rows, cannot give both children `min_samples_leaf` rows, reached `max_depth`, or no allowed split
exists. The tree is a dict of flat NumPy arrays, the same layout as scikit-learn's `tree_`:
`feature`, `threshold`, `left`, `right` (`-1` = leaf), `impurity`, `n_node_samples`, `weighted_n_node_samples`,
`value`.

> **Implementation hurdle — sample weights vs duplicated rows.** A bootstrap sample contains some rows 2, 3, 4…
> times. scikit-learn does not copy them; it passes the **counts as sample weights**. That is the same thing for
> impurities and leaf values, but not for rules that count *rows*: `min_samples_leaf=5` counts **distinct** rows
> when weights are used, and copies when rows are duplicated. The check below shows both effects.

**Check:** the 8-row example gives the README's tree (root `x1 <= 4.5`, right child `x2 <= 2.0`).

In [7]:
def fit_tree(X32, T, w, task, draw, max_depth=None, min_samples_split=2, min_samples_leaf=1):
    """Grow one CART tree on weighted rows. `draw(n_known_constants, is_constant)` picks the candidate features."""
    nodes = {k: [] for k in ["feature", "threshold", "left", "right", "impurity",
                             "n_node_samples", "weighted_n_node_samples", "value"]}
    max_depth = np.inf if max_depth is None else max_depth

    def grow(rows, depth, impurity, n_known_constants):
        tot = node_totals(T, w, rows, task)
        impurity = node_impurity(tot, task) if impurity is None else impurity
        node = len(nodes["feature"])
        for key, val in [("feature", -2), ("threshold", -2.0), ("left", -1), ("right", -1), ("impurity", impurity),
                         ("n_node_samples", len(rows)), ("weighted_n_node_samples", tot["w"]),
                         ("value", node_value(tot, task))]:
            nodes[key].append(val)
        if (depth >= max_depth or len(rows) < min_samples_split or len(rows) < 2 * min_samples_leaf
                or impurity <= EPSILON):
            return node
        Xn = X32[rows]
        is_constant = Xn.max(axis=0) <= Xn.min(axis=0) + FEATURE_THRESHOLD
        candidates, n_constants = draw(n_known_constants, is_constant)
        split = find_best_split(X32, T, w, rows, candidates, task, tot, min_samples_leaf) if candidates else None
        if split is None:
            return node
        nodes["feature"][node], nodes["threshold"][node] = split["feature"], split["threshold"]
        i_left = node_impurity(node_totals(T, w, split["left"], task), task)
        i_right = node_impurity(node_totals(T, w, split["right"], task), task)
        nodes["left"][node] = grow(split["left"], depth + 1, i_left, n_constants)
        nodes["right"][node] = grow(split["right"], depth + 1, i_right, n_constants)
        return node

    grow(np.flatnonzero(w > 0), 0, None, 0)                    # rows with weight 0 are not in the tree
    return {k: np.array(v) for k, v in nodes.items()}


def apply_tree(tree, X32):
    """Leaf number reached by each row (all rows move down one level per loop)."""
    node = np.zeros(len(X32), dtype=int)
    X64 = X32.astype(np.float64)
    while True:
        active = np.flatnonzero(tree["left"][node] != -1)
        if len(active) == 0:
            return node
        nd = node[active]
        go_left = X64[active, tree["feature"][nd]] <= tree["threshold"][nd]
        node[active] = np.where(go_left, tree["left"][nd], tree["right"][nd])


def predict_tree(tree, X32):
    return tree["value"][apply_tree(tree, X32)]


toy = fit_tree(X_toy, T_toy, w_toy, "classification", draw_all_features)
for key in ["feature", "threshold", "left", "right", "impurity", "weighted_n_node_samples"]:
    print(f"{key:24s}", toy[key])
print("value (class fractions):\n", toy["value"])

feature                  [ 0 -2  1 -2 -2]
threshold                [ 4.5 -2.   2.  -2.  -2. ]
left                     [ 1 -1  3 -1 -1]
right                    [ 2 -1  4 -1 -1]
impurity                 [0.4688 0.     0.375  0.     0.    ]
weighted_n_node_samples  [8. 4. 4. 1. 3.]
value (class fractions):
 [[0.625 0.375]
 [1.    0.   ]
 [0.25  0.75 ]
 [1.    0.   ]
 [0.    1.   ]]


In [8]:
rng = np.random.default_rng(0)
counts = np.bincount(rng.integers(0, len(Xg_tr), len(Xg_tr)), minlength=len(Xg_tr)).astype(float)  # one bootstrap
duplicated = np.repeat(np.arange(len(Xg_tr)), counts.astype(int))                                 # same, as copies
for leaf in [1, 5]:
    mine = fit_tree(Xg_tr32, yg_tr[:, None], counts, "regression", draw_all_features, min_samples_leaf=leaf)
    sk_w = DecisionTreeRegressor(min_samples_leaf=leaf, random_state=0).fit(Xg_tr32, yg_tr, sample_weight=counts)
    sk_dup = DecisionTreeRegressor(min_samples_leaf=leaf, random_state=0).fit(Xg_tr32[duplicated], yg_tr[duplicated])
    print(f"min_samples_leaf={leaf}: nodes  ours {len(mine['feature'])} | sklearn + weights {sk_w.tree_.node_count} | "
          f"sklearn + copies {sk_dup.tree_.node_count};   max |prediction(weights) - prediction(copies)| = "
          f"{np.abs(sk_w.predict(Xg_te32) - sk_dup.predict(Xg_te32)).max():.2f}")
print(f"rows at the root: ours {mine['n_node_samples'][0]} distinct rows with total weight "
      f"{mine['weighted_n_node_samples'][0]:.0f};  copies: {sk_dup.tree_.n_node_samples[0]} rows")

min_samples_leaf=1: nodes  ours 1173 | sklearn + weights 1173 | sklearn + copies 1173;   max |prediction(weights) - prediction(copies)| = 0.00
min_samples_leaf=5: nodes  ours 191 | sklearn + weights 191 | sklearn + copies 315;   max |prediction(weights) - prediction(copies)| = 1.39
rows at the root: ours 622 distinct rows with total weight 1000;  copies: 1000 rows


> **How to read this**
> * The 8-row tree has exactly the arrays printed from scikit-learn's `tree_` in Lab 2 of the library notebook:
>   root on feature 0 at 4.5, node 2 on feature 1 at 2.0, impurities 0.4688 / 0 / 0.375 / 0 / 0, class fractions
>   `[0.625, 0.375]` at the root.
> * Weights vs copies: with `min_samples_leaf=1` all three trees have 1173 nodes and weights and copies predict
>   identically (0.00). With `min_samples_leaf=5` the weighted trees (ours and scikit-learn's) have 191 nodes but the
>   tree on copied rows has 315, and predictions differ by up to 1.39: the root holds 622 *distinct* rows (weight 1000)
>   in one case and 1000 rows in the other, so "5 rows per leaf" means different things.
> * Our tree uses the same node numbering (node, left subtree, right subtree) as scikit-learn; whether it is the *same*
>   tree, split for split, is tested in section 9.

---
# Part B — The forest

## 6 · Bootstrap, averaging and the out-of-bag score (README §3 steps 1 and 5, §4.4)

* `bootstrap_counts`: draw $n$ row numbers with replacement; `np.bincount` turns them into "how many times was row
  $i$ drawn" — the weights for one tree.
* `fit_forest`: one tree per (counts, sampler) pair.
* `predict_forest`: regression = mean of the trees; classification = mean of the trees' class fractions, then the
  most likely class (scikit-learn's "soft vote").
* `oob_score`: for every training row, average only the trees whose counts are 0 for that row, then score
  (R² for regression, accuracy for classification — the same as scikit-learn's `oob_score_`).

In [9]:
def bootstrap_counts(rng, n):
    return np.bincount(rng.integers(0, n, n), minlength=n).astype(float)


def fit_forest(X32, T, task, counts_list, draws, **tree_params):
    return [fit_tree(X32, T, c, task, d, **tree_params) for c, d in zip(counts_list, draws)]


def predict_forest(trees, X32, task):
    mean = np.mean([predict_tree(t, X32) for t in trees], axis=0)
    return mean.argmax(axis=1) if task == "classification" else mean[:, 0]


def oob_score(trees, counts_list, X32, y, task):
    total, n_used = 0.0, np.zeros(len(X32))
    for tree, c in zip(trees, counts_list):
        out = c == 0
        pred = np.zeros((len(X32), tree["value"].shape[1]))
        pred[out] = predict_tree(tree, X32[out])
        total, n_used = total + pred, n_used + out
    mean = total[n_used > 0] / n_used[n_used > 0, None]
    if task == "classification":
        return accuracy_score(y[n_used > 0], mean.argmax(axis=1))
    return r2_score(y[n_used > 0], mean[:, 0])


def numpy_forest_inputs(seed, n_rows, n_trees, max_features):
    """Our own random numbers: one bootstrap and one feature sampler per tree."""
    rng = np.random.default_rng(seed)
    counts_list = [bootstrap_counts(rng, n_rows) for _ in range(n_trees)]
    draws = [make_numpy_sampler(rng, max_features) for _ in range(n_trees)]
    return counts_list, draws


counts_g, draws_g = numpy_forest_inputs(seed=0, n_rows=len(Xg_tr), n_trees=100, max_features=5)   # <- try 31
t0 = time.perf_counter()
forest_g = fit_forest(Xg_tr32, yg_tr[:, None], "regression", counts_g, draws_g)
print(f"100 trees in {time.perf_counter() - t0:.1f} s;  test MSE {mean_squared_error(yg_te, predict_forest(forest_g, Xg_te32, 'regression')):.3f}"
      f";  OOB R² {oob_score(forest_g, counts_g, Xg_tr32, yg_tr, 'regression'):.3f}")
print(f"rows per tree: {len(Xg_tr)} draws, {np.mean([np.sum(c > 0) for c in counts_g]):.0f} distinct on average "
      f"({np.mean([np.mean(c > 0) for c in counts_g]):.1%}, theory 63.2%)")
print("drawn features that were constant in their node (good fit, 4-decimal data):", sum(d.constant_draws for d in draws_g),
      "| extra draws beyond max_features:", sum(d.extra_draws for d in draws_g))

counts_b, draws_b = numpy_forest_inputs(seed=0, n_rows=len(Xb_tr), n_trees=100, max_features=12)
forest_b = fit_forest(Xb_tr32, np.eye(2)[yb_tr], "classification", counts_b, draws_b)
print(f"\nbad fit, 100 trees, max_features=12: test accuracy {accuracy_score(yb_te, predict_forest(forest_b, Xb_te32, 'classification')):.3f}"
      f";  OOB accuracy {oob_score(forest_b, counts_b, Xb_tr32, yb_tr, 'classification'):.3f}")
print("drawn features that were constant in their node (bad fit, 2-decimal data):", sum(d.constant_draws for d in draws_b),
      "| extra draws beyond max_features:", sum(d.extra_draws for d in draws_b))

100 trees in 2.6 s;  test MSE 0.301;  OOB R² 0.932
rows per tree: 1000 draws, 632 distinct on average (63.2%, theory 63.2%)
drawn features that were constant in their node (good fit, 4-decimal data): 4 | extra draws beyond max_features: 0



bad fit, 100 trees, max_features=12: test accuracy 0.750;  OOB accuracy 0.711
drawn features that were constant in their node (bad fit, 2-decimal data): 6 | extra draws beyond max_features: 0


> **How to read this**
> * Our 100-tree forest (5 candidates per split) reaches test MSE **0.301** and OOB R² **0.932** on the good fit —
>   in line with scikit-learn's forests in the library notebook (0.299 with 300 trees, OOB R² 0.935).
> * Each tree is trained on 1000 draws but only 632 distinct rows (63.2 %), as the bootstrap theory predicts.
> * Bad fit (12 candidates per split): accuracy 0.750, OOB accuracy 0.711.
> * Only 4 (good fit) and 6 (bad fit) drawn features turned out constant in their node, and no node ever needed extra
>   draws on these data. The lazy rule of section 4 is a safety net for small nodes of coarse data, not a frequent
>   event.

## 7 · MDI feature importance (README §4.6)

> **Implementation hurdle — MDI bookkeeping.** Three things must be right, and the old notes got the first two wrong:
> (1) add each split's **weighted impurity decrease**
> $W_t\,i_t - W_L\,i_L - W_R\,i_R$ (divided by $W_\text{root}$), not the node's raw impurity;
> (2) add it to the feature **actually used** (`feature[t]`, not `feature[t] - 1`);
> (3) normalise **each tree** to sum 1, then average the trees (and renormalise).

**Check:** the 8-row tree gives 0.6 / 0.4, as computed by hand in README §5.

In [10]:
def tree_mdi(tree, n_features):
    importance = np.zeros(n_features)
    for t in np.flatnonzero(tree["left"] != -1):
        L, R = tree["left"][t], tree["right"][t]
        W, imp = tree["weighted_n_node_samples"], tree["impurity"]
        importance[tree["feature"][t]] += W[t] * imp[t] - W[L] * imp[L] - W[R] * imp[R]
    importance /= tree["weighted_n_node_samples"][0]
    return importance / importance.sum() if importance.sum() > 0 else importance


def forest_mdi(trees, n_features):
    mean = np.mean([tree_mdi(t, n_features) for t in trees if len(t["feature"]) > 1], axis=0)
    return mean / mean.sum()


def tree_mdi_old_buggy(tree, n_features):
    importance = np.zeros(n_features)
    for t in np.flatnonzero(tree["left"] != -1):
        importance[tree["feature"][t] - 1] += tree["impurity"][t]
    return importance / importance.sum()


print("8-row tree   correct MDI:", tree_mdi(toy, 2), "   old buggy version:", tree_mdi_old_buggy(toy, 2))
mdi_g = forest_mdi(forest_g, 31)
print("good-fit forest: MDI of x0 =", mdi_g[0].round(3), "; mean MDI of a cheap thermometer =", mdi_g[1:].mean().round(3))

8-row tree   correct MDI: [0.6 0.4]    old buggy version: [0.4444 0.5556]
good-fit forest: MDI of x0 = 0.209 ; mean MDI of a cheap thermometer = 0.026


> **How to read this**
> * The 8-row tree: **0.6 / 0.4**, the README's hand result; the old version gives 0.444 / 0.556 and swaps the ranking.
> * On the good fit our forest gives the precise thermometer x0 an MDI of 0.209 and each cheap one 0.026 on average —
>   the same picture as scikit-learn's forest in Lab 15 (0.168 / 0.028; different number of trees and random draws).

## 8 · Implementation hurdles: cost per split and memory of deep trees

* **Cost.** Searching one node costs about $m \cdot n\log n$ for $m$ candidate features — so `max_features` is also
  a speed knob. We time the root split of the bad-fit data for several $m$, and whole trees.
* **Memory.** A fully grown tree keeps splitting until nodes are pure or hold one distinct row, so it has about
  $2\times$(distinct rows) $-1$ nodes; a forest stores `n_estimators` of them.

In [11]:
w1 = np.ones(len(Xb_tr))
T_bad = np.eye(2)[yb_tr]
tot_root = node_totals(T_bad, w1, np.arange(len(Xb_tr)), "classification")
print("  m   root split [ms]   whole tree [ms]")
for m in [1, 12, 50, 150]:                                         # <- try changing this
    cand = list(range(m))
    t0 = time.perf_counter()
    for _ in range(5):
        find_best_split(Xb_tr32, T_bad, w1, np.arange(len(Xb_tr)), cand, "classification", tot_root)
    t_root = (time.perf_counter() - t0) / 5
    t0 = time.perf_counter()
    fit_tree(Xb_tr32, T_bad, bootstrap_counts(np.random.default_rng(0), len(Xb_tr)), "classification",
             make_numpy_sampler(np.random.default_rng(1), m))
    print(f"{m:4d}   {1e3 * t_root:12.2f}     {1e3 * (time.perf_counter() - t0):12.0f}")

nodes = np.array([len(t["feature"]) for t in forest_g])
distinct = np.array([np.sum(c > 0) for c in counts_g])
print(f"\ngood-fit trees: {nodes.mean():.0f} nodes on average for {distinct.mean():.0f} distinct rows "
      f"(2 x distinct - 1 = {2 * distinct.mean() - 1:.0f})")
bytes_ours = sum(sum(v.nbytes for v in t.values()) for t in forest_g)
print(f"our 100 trees use {bytes_ours / 1e6:.1f} MB; scikit-learn stores 64 bytes per node + 8 per output value "
      f"≈ {nodes.sum() * 72 / 1e6:.1f} MB.  With 1 million rows: ≈ {2 * 0.632e6 * 72 * 100 / 1e9:.0f} GB for 100 trees.")

  m   root split [ms]   whole tree [ms]
   1           0.10                7
  12           0.43                5
  50           1.89               10
 150           5.72               31

good-fit trees: 1224 nodes on average for 632 distinct rows (2 x distinct - 1 = 1263)
our 100 trees use 7.8 MB; scikit-learn stores 64 bytes per node + 8 per output value ≈ 8.8 MB.  With 1 million rows: ≈ 9 GB for 100 trees.


> **How to read this**
> * Root split: well under 1 ms with 1 or 12 candidates, ≈ 2 ms with 50, ≈ 6 ms with 150 — roughly proportional to
>   m (plus a fixed overhead). Whole trees: under ≈ 10 ms for m ≤ 50, ≈ 30 ms for m = 150. (With m = 1 the splits are poor, so the tree grows more nodes; that is why its
>   whole-tree time is not the smallest.)
> * A good-fit tree has 1224 nodes for 632 distinct rows — close to 2 × 632 − 1 = 1263. Our 100 trees take 7.8 MB,
>   scikit-learn's layout ≈ 8.8 MB; extrapolated to a million rows, 100 deep trees would need ≈ 9 GB (Lab 16 of the
>   library notebook measures 176 MB at 20,000 rows).

---
# Part C — Checking against scikit-learn

## 9 · Bit-exact agreement in the bagging limit (`max_features=None`)

With `max_features=None` there is no feature sampling, so the only randomness is the bootstrap. scikit-learn
exposes its draws as `estimators_samples_`; `np.bincount` turns them into the counts it passed to each tree as
`sample_weight` (see `_parallel_build_trees` in `sklearn/ensemble/_forest.py`). Feeding the **same counts** to our
`fit_tree` should give the same trees.

**First attempt:** our tree evaluates the features in index order 0, 1, 2, ….

In [12]:
def compare_trees(sk_forest, my_trees, X_test32):
    """Per tree: identical structure? largest prediction difference. Plus the importance difference."""
    same, worst, first_bad = 0, 0.0, None
    for b, (est, mine) in enumerate(zip(sk_forest.estimators_, my_trees)):
        t = est.tree_
        identical = (t.node_count == len(mine["feature"]) and np.array_equal(t.feature, mine["feature"])
                     and np.array_equal(t.threshold, mine["threshold"]))
        diff = np.abs(t.value[est.apply(X_test32)][:, 0, :] - predict_tree(mine, X_test32)).max()
        same += identical
        worst = max(worst, diff)
        if not identical and first_bad is None:
            first_bad = b
    imp = np.abs(forest_mdi(my_trees, X_test32.shape[1]) - sk_forest.feature_importances_).max()
    return same, worst, imp, first_bad


def sklearn_counts(sk_forest, n_rows):
    return [np.bincount(idx, minlength=n_rows).astype(float) for idx in sk_forest.estimators_samples_]


sk_bag_g = RandomForestRegressor(100, max_features=None, random_state=0).fit(Xg_tr, yg_tr)
sk_bag_b = RandomForestClassifier(100, max_features=None, random_state=0).fit(Xb_tr, yb_tr)
first_try = {}
for name, sk, X32, T, task, X_te32 in [
        ("good fit (regression)", sk_bag_g, Xg_tr32, yg_tr[:, None], "regression", Xg_te32),
        ("bad fit (classification)", sk_bag_b, Xb_tr32, np.eye(2)[yb_tr], "classification", Xb_te32)]:
    cl = sklearn_counts(sk, len(X32))
    trees = fit_forest(X32, T, task, cl, [draw_all_features] * 100)
    same, worst, imp, first_bad = compare_trees(sk, trees, X_te32)
    first_try[name] = (sk, trees, cl, X32, T, task, first_bad)
    print(f"{name:25s}: identical trees {same}/100, max |prediction diff| {worst:.3g}, max |importance diff| {imp:.2g}")

good fit (regression)    : identical trees 0/100, max |prediction diff| 3.81, max |importance diff| 0.0045


bad fit (classification) : identical trees 0/100, max |prediction diff| 1, max |importance diff| 0.02


Some trees differ. Look at the first node where one of them disagrees: both features give **exactly** the same
split quality — a tie. Ties are common in small nodes (e.g. two features that separate the same 3 rows from the
other 4, or mirror images of each other). scikit-learn keeps the **first** best feature it evaluates, and it
evaluates features in a random order even when it uses all of them.

In [13]:
def first_disagreement(sk_est, mine):
    t = sk_est.tree_
    m = min(t.node_count, len(mine["feature"]))
    return int(np.flatnonzero((t.feature[:m] != mine["feature"][:m]) | (t.threshold[:m] != mine["threshold"][:m]))[0])


sk, trees, cl, X32, T, task, b = first_try["bad fit (classification)"]
node = first_disagreement(sk.estimators_[b], trees[b])
rows_in_node = np.flatnonzero(sk.estimators_[b].decision_path(X32)[:, node].toarray().ravel() & (cl[b] > 0))
tot = node_totals(T, cl[b], rows_in_node, task)
for f in [trees[b]["feature"][node], sk.estimators_[b].tree_.feature[node]]:
    s = find_best_split(X32, T, cl[b], rows_in_node, [int(f)], task, tot)
    dec = weighted_decrease(tot, node_totals(T, cl[b], s["left"], task), node_totals(T, cl[b], s["right"], task), task, cl[b].sum())
    print(f"tree {b}, node {node} ({len(rows_in_node)} rows): feature x{f:<3d} threshold {s['threshold']:+.3f}  "
          f"weighted decrease {dec:.17f}")

tree 0, node 9 (3 rows): feature x0   threshold -0.595  weighted decrease 0.00267857142857143
tree 0, node 9 (3 rows): feature x53  threshold -0.065  weighted decrease 0.00267857142857143


> **Implementation hurdle — ties and the order in which features are tried.** To agree tree for tree we must try
> features in scikit-learn's order. That order comes from a tiny random-number generator inside its Cython splitter
> (`_splitter.pyx`): a 32-bit *xorshift* generator, seeded once per tree with
> `RandomState(tree.random_state).randint(0, 2**31 - 1)`, driving a Fisher–Yates shuffle of the feature list that
> carries over from node to node and remembers features already known to be constant. `make_sklearn_sampler` is a
> line-by-line transcription of that loop. It is the only part of this notebook that copies an implementation
> detail rather than the algorithm — the forest would be just as good with any random order (section 10 shows that);
> we need it only to prove that everything else is identical.
>
> **Implementation hurdle — the last bit.** The good-fit target is a multiple of 1/64. Then every weighted sum of
> targets is exact in floating point, so two splits that are mathematically tied are tied to the last bit in both
> implementations. With an arbitrary target, sums taken in a different order differ by ~1e-16, and which of two
> tied splits "wins" becomes a coin toss decided by rounding. (The last cell of section 9 demonstrates it.)

In [14]:
RAND_R_MAX = 2**31 - 1


def make_sklearn_sampler(tree_seed, n_features, max_features):
    """Replays scikit-learn's per-node feature draws (node_split_best in sklearn/tree/_splitter.pyx)."""
    state = {"seed": int(np.random.RandomState(tree_seed).randint(0, RAND_R_MAX)),
             "features": np.arange(n_features), "constant_features": np.zeros(n_features, dtype=int)}

    def rand_int(low, high):                                    # xorshift32, then modulo
        s = state["seed"] or 1
        s ^= (s << 13) & 0xFFFFFFFF
        s ^= s >> 17
        s ^= (s << 5) & 0xFFFFFFFF
        state["seed"] = s
        return low + (s % (RAND_R_MAX + 1)) % (high - low)

    def draw(n_known_constants, is_constant):
        features, constant_features = state["features"], state["constant_features"]
        f_i, n_visited, n_found, n_drawn = n_features, 0, 0, 0
        n_total = n_known_constants
        candidates = []
        while f_i > n_total and (n_visited < max_features or n_visited <= n_found + n_drawn):
            n_visited += 1
            f_j = rand_int(n_drawn, f_i - n_found)
            if f_j < n_known_constants:                         # drew a feature already known to be constant
                features[[n_drawn, f_j]] = features[[f_j, n_drawn]]
                n_drawn += 1
                continue
            f_j += n_found
            if is_constant[features[f_j]]:                      # newly found constant feature
                features[[f_j, n_total]] = features[[n_total, f_j]]
                n_found += 1
                n_total += 1
                continue
            f_i -= 1                                            # a real candidate
            features[[f_i, f_j]] = features[[f_j, f_i]]
            candidates.append(int(features[f_i]))
        features[:n_known_constants] = constant_features[:n_known_constants]
        constant_features[n_known_constants:n_total] = features[n_known_constants:n_total]
        return candidates, n_total

    return draw


exact = {}
for name, (sk, _, cl, X32, T, task, _) in first_try.items():
    X_te32 = Xg_te32 if task == "regression" else Xb_te32
    draws = [make_sklearn_sampler(est.random_state, X32.shape[1], est.max_features_) for est in sk.estimators_]
    t0 = time.perf_counter()
    trees = fit_forest(X32, T, task, cl, draws)
    same, worst, imp, _ = compare_trees(sk, trees, X_te32)
    exact[name] = trees
    print(f"{name:25s}: identical trees {same}/100, max |prediction diff| {worst:.3g}, "
          f"max |importance diff| {imp:.2g}  ({time.perf_counter() - t0:.1f} s)")

good fit (regression)    : identical trees 100/100, max |prediction diff| 0, max |importance diff| 4.3e-18  (5.3 s)


bad fit (classification) : identical trees 100/100, max |prediction diff| 0, max |importance diff| 2.8e-17  (2.7 s)


In [15]:
# the last bit: the same good-fit data with a target that is NOT a multiple of 1/64
y_third = yg_tr / 3
sk_third = RandomForestRegressor(30, max_features=None, random_state=0).fit(Xg_tr, y_third)
draws = [make_sklearn_sampler(est.random_state, 31, est.max_features_) for est in sk_third.estimators_]
trees = fit_forest(Xg_tr32, y_third[:, None], "regression", sklearn_counts(sk_third, len(Xg_tr)), draws)
same, worst, imp, _ = compare_trees(sk_third, trees, Xg_te32)
print(f"target y/3: identical trees {same}/30, max |prediction diff| {worst:.3g}")

target y/3: identical trees 0/30, max |prediction diff| 0.833


> **How to read this**
> * **First attempt (features tried in index order): 0 of 100 trees identical** on either dataset, with prediction
>   differences up to 3.81 (regression) and whole class flips (classification).
> * The diagnosis shows why: in tree 0, node 9 holds 3 rows, and x0 and x53 give **exactly** the same weighted decrease
>   (0.00267857…). We kept x0 because it comes first in index order; scikit-learn kept x53 because its random order
>   reached x53 first. One different choice changes the whole subtree below it.
> * **Replaying scikit-learn's order: 100 of 100 trees identical** on both datasets — same features, same thresholds,
>   prediction difference **0** (bit-for-bit) and importance difference ≤ 3e-17 (rounding in the 17th digit, from
>   summing in a different order).
> * The `y/3` check: the same data with a target that is not a multiple of 1/64 gives 0 of 30 identical trees
>   (differences up to 0.83). Tied splits are then decided by last-bit rounding of sums taken in a different order —
>   which is why the dataset's target is stored as multiples of 1/64.

## 10 · With feature sampling: our random numbers vs scikit-learn's

Now the real random forest: `max_features=5` on the good fit (31 features) and `'sqrt'` = 12 on the bad fit
(150 features), 100 trees, all other settings at their defaults, on both sides.

* **(a) Our own random numbers** (`numpy_forest_inputs`). Bootstraps and feature draws come from a different
  random-number stream, so the trees are different trees — exact agreement is impossible and not the goal. A correct
  implementation must agree **statistically**: similar test metric, similar OOB score, the same important features,
  and highly similar predictions.
* **(b) Replaying scikit-learn's random numbers** (`sklearn_counts` + `make_sklearn_sampler`): the same trees,
  tree for tree.

In [16]:
setups = [("good fit", RandomForestRegressor, 5, Xg_tr, Xg_tr32, yg_tr, yg_tr[:, None], Xg_te32, yg_te, "regression"),
          ("bad fit", RandomForestClassifier, "sqrt", Xb_tr, Xb_tr32, yb_tr, np.eye(2)[yb_tr], Xb_te32, yb_te, "classification")]
results = []
for name, Model, mf, X_tr, X32, y_tr, T, X_te32, y_te, task in setups:
    t0 = time.perf_counter()
    sk = Model(100, max_features=mf, oob_score=True, random_state=0).fit(X_tr, y_tr)
    t_sk = time.perf_counter() - t0
    m = sk.estimators_[0].max_features_
    cl_np, dr_np = numpy_forest_inputs(seed=0, n_rows=len(X32), n_trees=100, max_features=m)
    t0 = time.perf_counter()
    ours = fit_forest(X32, T, task, cl_np, dr_np)
    t_ours = time.perf_counter() - t0
    p_sk, p_ours = sk.predict(X_te32), predict_forest(ours, X_te32, task)
    score = (lambda p: accuracy_score(y_te, p)) if task == "classification" else (lambda p: mean_squared_error(y_te, p))
    agree = np.mean(p_sk == p_ours) if task == "classification" else np.corrcoef(p_sk, p_ours)[0, 1]
    sk_other = Model(100, max_features=mf, oob_score=True, random_state=1).fit(X_tr, y_tr)   # yardstick
    p_other = sk_other.predict(X_te32)
    agree_other = np.mean(p_sk == p_other) if task == "classification" else np.corrcoef(p_sk, p_other)[0, 1]
    imp_ours = forest_mdi(ours, X32.shape[1])
    top_sk, top_ours = np.argsort(sk.feature_importances_)[::-1][:5], np.argsort(imp_ours)[::-1][:5]
    replay = fit_forest(X32, T, task, sklearn_counts(sk, len(X32)),
                        [make_sklearn_sampler(e.random_state, X32.shape[1], m) for e in sk.estimators_])
    same, worst, imp_diff, _ = compare_trees(sk, replay, X_te32)
    results.append((name, task, m, score(p_sk), score(p_ours), score(p_other), sk.oob_score_,
                    oob_score(ours, cl_np, X32, y_tr, task), sk_other.oob_score_, agree, agree_other,
                    np.corrcoef(sk.feature_importances_, imp_ours)[0, 1],
                    np.corrcoef(sk.feature_importances_, sk_other.feature_importances_)[0, 1],
                    top_sk, top_ours, t_sk, t_ours, same, worst, imp_diff, sk, replay))

for (name, task, m, s_sk, s_ours, s_other, oob_sk, oob_ours, oob_other, agree, agree_other, imp_corr, imp_corr_other,
     top_sk, top_ours, t_sk, t_ours, same, worst, imp_diff, _, _) in results:
    metric = "accuracy" if task == "classification" else "test MSE"
    agree_txt = "share of identical class predictions" if task == "classification" else "correlation of test predictions"
    print(f"--- {name} ({task}, max_features={m}, 100 trees) ---")
    print(f"(a)                          sklearn seed 0 | ours (NumPy RNG) | sklearn seed 1 (yardstick)")
    print(f"    {metric:24s} {s_sk:10.3f}     | {s_ours:10.3f}       | {s_other:10.3f}")
    print(f"    {'OOB score':24s} {oob_sk:10.3f}     | {oob_ours:10.3f}       | {oob_other:10.3f}")
    print(f"    {agree_txt + ' with seed 0':42s}  {agree:.3f}  vs yardstick {agree_other:.3f}")
    print(f"    {'correlation of MDI importances with seed 0':42s}  {imp_corr:.3f}  vs yardstick {imp_corr_other:.3f}")
    print(f"    top-5 features: sklearn {top_sk} | ours {top_ours}")
    print(f"    fit time: sklearn {t_sk:.2f} s (compiled Cython) | ours {t_ours:.1f} s (NumPy + Python)")
    print(f"(b) replayed random numbers: identical trees {same}/100, max |prediction diff| {worst:.3g}, "
          f"max |importance diff| {imp_diff:.2g}\n")

--- good fit (regression, max_features=5, 100 trees) ---
(a)                          sklearn seed 0 | ours (NumPy RNG) | sklearn seed 1 (yardstick)
    test MSE                      0.306     |      0.301       |      0.294
    OOB score                     0.932     |      0.932       |      0.930
    correlation of test predictions with seed 0  0.999  vs yardstick 0.998
    correlation of MDI importances with seed 0  0.941  vs yardstick 0.900
    top-5 features: sklearn [ 0  6  2 24 30] | ours [ 0  2  6 30 24]
    fit time: sklearn 0.17 s (compiled Cython) | ours 2.6 s (NumPy + Python)
(b) replayed random numbers: identical trees 100/100, max |prediction diff| 0, max |importance diff| 2.8e-17

--- bad fit (classification, max_features=12, 100 trees) ---
(a)                          sklearn seed 0 | ours (NumPy RNG) | sklearn seed 1 (yardstick)
    accuracy                      0.762     |      0.750       |      0.746
    OOB score                     0.727     |      0.711       | 

The few replayed trees that still differ on the bad fit: at the first differing node, compare scikit-learn's chosen
split with ours. `split_proxy` computes the Gini proxy $-W_R\,i_R - W_L\,i_L$ of a given (feature, threshold) in
plain arithmetic, and once more the way compiled code may evaluate it — with a **fused multiply-add**
(`a*b + c` rounded once instead of twice; `fused_multiply_add` does it exactly with Python's integer arithmetic).

In [17]:
def fused_multiply_add(a, b, c):
    """a*b + c with a single rounding, computed exactly from the floats' integer ratios."""
    (na, da), (nb, db), (nc, dc) = a.as_integer_ratio(), b.as_integer_ratio(), c.as_integer_ratio()
    return (na * nb * dc + nc * da * db) / (da * db * dc)


def split_proxy(X32, T, w, rows, feature, threshold):
    """Gini proxy of one given split: plain arithmetic and fused multiply-add."""
    go_left = X32[rows, feature].astype(np.float64) <= threshold
    L = node_totals(T, w, rows[go_left], "classification")
    R = node_totals(T, w, rows[~go_left], "classification")
    gL, gR = gini_impurity(L["counts"], L["w"]), gini_impurity(R["counts"], R["w"])
    return -R["w"] * gR - L["w"] * gL, fused_multiply_add(float(-R["w"]), float(gR), float(-(L["w"] * gL)))


name, task, *_, sk, replay = results[1]
cl = sklearn_counts(sk, len(Xb_tr32))
T_b = np.eye(2)[yb_tr]
for b, (est, mine) in enumerate(zip(sk.estimators_, replay)):
    if est.tree_.node_count == len(mine["feature"]) and np.array_equal(est.tree_.threshold, mine["threshold"]):
        continue
    node = first_disagreement(est, mine)
    rows_in_node = np.flatnonzero(est.decision_path(Xb_tr32)[:, node].toarray().ravel() & (cl[b] > 0))
    ours_split = (int(mine["feature"][node]), mine["threshold"][node])
    sk_split = (int(est.tree_.feature[node]), est.tree_.threshold[node])
    plain_o, fused_o = split_proxy(Xb_tr32, T_b, cl[b], rows_in_node, *ours_split)
    plain_s, fused_s = split_proxy(Xb_tr32, T_b, cl[b], rows_in_node, *sk_split)
    print(f"tree {b:2d} node {node:3d}: ours x{ours_split[0]}<= {ours_split[1]:+.3f}, sklearn x{sk_split[0]}<= {sk_split[1]:+.3f} | "
          f"plain: tie {plain_o == plain_s} | fused: sklearn's is larger {fused_s > fused_o} (by {fused_s - fused_o:.1e})")

tree  1 node  42: ours x139<= +0.575, sklearn x71<= -0.820 | plain: tie True | fused: sklearn's is larger True (by 1.8e-15)
tree 15 node 111: ours x108<= +0.865, sklearn x111<= -0.480 | plain: tie True | fused: sklearn's is larger True (by 8.9e-16)
tree 75 node  72: ours x140<= -0.315, sklearn x140<= -0.240 | plain: tie True | fused: sklearn's is larger True (by 1.8e-15)
tree 77 node  80: ours x12<= +1.020, sklearn x16<= -1.460 | plain: tie True | fused: sklearn's is larger True (by 1.8e-15)
tree 82 node 123: ours x16<= -1.275, sklearn x147<= +1.300 | plain: tie True | fused: sklearn's is larger True (by 8.9e-16)


> **How to read this**
> * **(a) Our random numbers.** Good fit: test MSE 0.306 (scikit-learn) vs **0.301** (ours) vs 0.294 (scikit-learn
>   with another seed); OOB R² 0.932 / 0.932 / 0.930; predictions correlate 0.999 (yardstick 0.998); importances
>   correlate 0.941 (yardstick 0.900); both top-5 lists start with x0. Bad fit: accuracy 0.762 / **0.750** / 0.746;
>   OOB 0.727 / 0.711 / 0.713; 83.8 % identical class predictions (yardstick 83.3 %); importances correlate 0.983
>   (0.978); top 3 = x2, x1, x0 in both. **Our forest differs from scikit-learn's exactly as much as scikit-learn's
>   differs from itself with another seed** — that is what "statistically the same algorithm" looks like.
> * Fit time: about 0.2 s for scikit-learn (compiled Cython, one core) vs ~2.5 s (good fit) and ~0.5 s (bad fit) for
>   ours — the price of a Python loop over nodes.
> * **(b) Replayed random numbers.** Good fit: 100/100 identical trees, prediction difference 0. Bad fit: 95/100. The
>   last cell examines the 5 that differ: each first differs at a node where two splits are **exactly tied** in plain
>   arithmetic, and in every case scikit-learn's choice is the larger one by ~1e-15 when the Gini proxy is evaluated
>   with a fused multiply-add — the compiled code on this machine rounds `a*b + c` once, which breaks the tie. This is
>   the last bit of floating-point arithmetic, not a difference in the algorithm (and it may not happen on a
>   different CPU or compiler).

---
# Summary — what each section showed

| Section | Theory step (README §) | Check / implementation hurdle | Verdict |
|---|---|---|---|
| 2 | Gini and MSE impurity (§2.2) | 8-row example: 0.46875 and 0.375 | ✔ exact |
| 3 | Best split by weighted impurity decrease (§2.3) | README §5 splits and decreases; naive O(n²) vs sort + cumsum (≈ 360 ms vs ≈ 1.2 ms at 16k rows) | ✔ |
| 4 | Per-split feature sampling (§3) | lazy drawing past `max_features` when drawn features are constant | ✔ never returns empty |
| 5 | One weighted tree (§3) | README tree reproduced; weights ≠ copies once `min_samples_leaf > 1` | ✔ |
| 6 | Bootstrap, averaging, OOB (§3, §4.4) | 63.2 % distinct rows; OOB R² 0.932 | ✔ |
| 7 | MDI (§4.6) | 0.6 / 0.4 by hand; the old version swaps the ranking | ✔ |
| 8 | Cost and memory (§6.4, §8.6) | split cost ∝ m; nodes ≈ 2 × distinct rows | ✔ |
| 9 | Bagging limit vs scikit-learn | ties need scikit-learn's feature order; then 100/100 identical trees, difference 0 | ✔ bit-exact |
| 10 | Random forest vs scikit-learn | own RNG: as close as scikit-learn with another seed; replayed RNG: 100/100 and 95/100 identical (the rest are last-bit ties) | ✔ |